In [120]:
import requests
import pandas as pd

In [121]:
import sys
sys.path.append("..")

from API_Key import gnews_API

In [122]:
queries = [
    "SAP",
    "SAP AI",
    "SAP ERP",
    "enterprise software",
    "business AI"
    "SAP Technologies"
]

In [123]:
API_KEY = gnews_API

In [124]:
import time

In [125]:
all_articles = []

for q in queries:

    url = (
        f"https://gnews.io/api/v4/search?"
        f"q={q}&lang=en&max=10&apikey={API_KEY}"
    )

    response = requests.get(url)
    time.sleep(0.2)
    data = response.json()

    if "articles" in data:
        all_articles.extend(data["articles"])
    else:
        print(f"Query failed: {q}")
        print(data)

In [126]:
len(all_articles)

32

In [127]:
gnews_docs = []

for article in all_articles:

    gnews_docs.append({
        "title": article["title"],
        "description": article["description"],
        "content": article["content"],
        "source": article["source"]["name"],
        "published": article["publishedAt"],
        "url": article["url"]
    })



In [128]:
gnews_df = pd.DataFrame(gnews_docs)

len(gnews_df)

32

Removing Duplicates

In [129]:
print("Before:", len(gnews_df))

gnews_df = gnews_df.drop_duplicates(
    subset=["title"]
)

print("After:", len(gnews_df))

Before: 32
After: 25


In [130]:
gnews_df.head(1)

,title,description,content,source,published,url
0,Highbar Technocrat Drives Digital Transformati...,"Highbar Technocrat Limited, a leading provider...","Highbar Technocrat Limited, a renowned technol...",Devdiscourse,2026-06-15T14:49:56Z,https://www.devdiscourse.com/article/business/...


In [131]:
from rapidfuzz import fuzz

titles = gnews_df["title"].fillna("").tolist()

to_drop = set()

print("Before:", len(gnews_df))
for i in range(len(titles)):
    if i in to_drop:
        continue

    for j in range(i + 1, len(titles)):
        score = fuzz.ratio(titles[i], titles[j])

        if score >= 80:
            to_drop.add(j)

gnews_df = gnews_df.drop(index=list(to_drop)).reset_index(drop=True)

print("After :", len(gnews_df))

Before: 25
After : 23


In [ ]:
gnews_df.to_json(
    r"C:\Users\mehat\OneDrive - SRH\Subject\NLP\data\gnews_articles.json",
    orient="records",
    indent=2
)